In [1]:
import librosa
import numpy as np
import torch
import torch.nn.functional as F
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

d:\Projects\Kaldi\Minimal_POC\.kaldi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
AUDIO_FILE = "input1.wav"
SILENCE_THRESHOLD = 0.01
MIN_SILENCE_SEC = 0.8

In [16]:
import librosa
import numpy as np

y, sr = librosa.load("input1.wav", sr=16000)

hop_length = 512
rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]

SILENCE_THRESHOLD = 0.02
MIN_SILENCE_SEC = 0.5

silent_frames = rms < SILENCE_THRESHOLD
segments = []

min_silence_frames = int(MIN_SILENCE_SEC * sr / hop_length)

start = 0
i = 0

while i < len(silent_frames):
    if silent_frames[i]:
        silence_start = i
        while i < len(silent_frames) and silent_frames[i]:
            i += 1
        silence_end = i
        
        if silence_end - silence_start > min_silence_frames:
            end = silence_start * hop_length
            if end > start:
                segments.append((start, end))
            start = silence_end * hop_length
    else:
        i += 1

if start < len(y):
    segments.append((start, len(y)))

print("Detected segments:", len(segments))

Detected segments: 9


In [17]:
# -----------------------------
# LOAD MODEL
# -----------------------------
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
model.eval()

d:\Projects\Kaldi\Minimal_POC\.kaldi\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhijith\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 1213.10it/s, Materializing param=wav2vec2.fea

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [18]:
# -----------------------------
# PROCESS EACH WORD
# -----------------------------
for idx, (s, e) in enumerate(segments):
    word_audio = y[s:e]

    if len(word_audio) < 400:
        continue

    inputs = processor(word_audio, sampling_rate=16000, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(inputs.input_values).logits

    # -----------------------------
    # DECODE TRANSCRIPTION
    # -----------------------------
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    # -----------------------------
    # CONFIDENCE ESTIMATION
    # -----------------------------
    probs = F.softmax(logits, dim=-1)

    # Frame-level max probability
    frame_confidence = torch.max(probs, dim=-1).values

    # Average confidence across frames
    avg_confidence = torch.mean(frame_confidence).item()

    # Entropy (uncertainty indicator)
    entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
    avg_entropy = torch.mean(entropy).item()

    print(f"\nWord {idx+1}:")
    print(f"Approx transcription: {transcription}")
    print(f"Confidence score: {avg_confidence:.3f}")
    print(f"Uncertainty (entropy): {avg_entropy:.3f}")


Word 1:
Approx transcription: SHI
Confidence score: 0.910
Uncertainty (entropy): 0.268

Word 2:
Approx transcription: SHE REIN
Confidence score: 0.845
Uncertainty (entropy): 0.514

Word 3:
Approx transcription: SHANT BE
Confidence score: 0.904
Uncertainty (entropy): 0.325

Word 4:
Approx transcription: HE
Confidence score: 0.954
Uncertainty (entropy): 0.217

Word 5:
Approx transcription: SSHOCKING THE
Confidence score: 0.948
Uncertainty (entropy): 0.148

Word 6:
Approx transcription: SHEEK
Confidence score: 0.928
Uncertainty (entropy): 0.231

Word 7:
Approx transcription: CHAR
Confidence score: 0.900
Uncertainty (entropy): 0.296

Word 8:
Approx transcription: SIGHING
Confidence score: 0.895
Uncertainty (entropy): 0.342

Word 9:
Approx transcription: SIMI
Confidence score: 0.924
Uncertainty (entropy): 0.268
